# Analytical PIAB2D eigenfunctions

**Prerequisite:** [Analytical PIAB2D eigenfunctions](../../../../../docs/lecture-notes/qm/piab2d/tise/analytical/piab2d__eigenfunctions.md)

This laboratory evaluates normalized product eigenfunctions and probability densities for the nondimensional unit square with homogeneous Dirichlet boundaries.

## Learning objectives

1. Evaluate selected $(n_x,n_y)$ stationary states on a rectangular coordinate grid.
2. Verify boundary zeros and two-dimensional normalization.
3. Relate quantum numbers to nodal structure.
4. Visualize probability-density heatmaps without embedding reusable evaluation code in the notebook.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab2d.base import Piab2D
from projectkoios.physkit.qm.piab2d.tise.analytical.eigenfunctions import (
    Piab2DAnalyticalEigenfunction,
)
from projectkoios.physkit.units import Unitless, UnitSystem, VectorQuantity

In [ ]:
unit_box_length = 1.0
unit_particle_mass = 1.0
model = Piab2D(
    length_x=unit_box_length,
    length_y=unit_box_length,
    mass=unit_particle_mass,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
sample_count = 201
coordinate_values = np.linspace(
    start=0.0,
    stop=unit_box_length,
    num=sample_count,
    dtype=np.float64,
)
coordinates = VectorQuantity(
    magnitude=coordinate_values,
    unit=Unitless(),
)
ground_quantum_number = 1
second_quantum_number = 2
selected_states = (
    (ground_quantum_number, ground_quantum_number),
    (ground_quantum_number, second_quantum_number),
    (second_quantum_number, ground_quantum_number),
    (second_quantum_number, second_quantum_number),
)

eigenfunction_evaluations = [
    Piab2DAnalyticalEigenfunction(
        model=model, quantum_number_x=n_x, quantum_number_y=n_y
    ).evaluate(coordinates_x=coordinates, coordinates_y=coordinates)
    for n_x, n_y in selected_states
]

In [ ]:
represented_norms = []
for result in eigenfunction_evaluations:
    density = result.probability_density.magnitude
    integrated_y = np.trapezoid(density, x=coordinate_values, axis=1)
    represented_norms.append(float(np.trapezoid(integrated_y, x=coordinate_values)))
    assert np.allclose(result.amplitudes.magnitude[[0, -1], :], 0.0)
    assert np.allclose(result.amplitudes.magnitude[:, [0, -1]], 0.0)

expected_norm = 1.0
normalization_tolerance = 2e-15
assert np.allclose(
    represented_norms,
    expected_norm,
    rtol=0.0,
    atol=normalization_tolerance,
)
represented_norms

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)

for axis, result in zip(axes.ravel(), eigenfunction_evaluations, strict=True):
    image = axis.pcolormesh(
        coordinate_values,
        coordinate_values,
        result.probability_density.magnitude.T,
        shading="auto",
        cmap="viridis",
    )
    axis.set(
        xlabel="$x/L_x$",
        ylabel="$y/L_y$",
        title=rf"$(n_x,n_y)=({result.quantum_number_x},{result.quantum_number_y})$",
        aspect="equal",
    )
    figure.colorbar(image, ax=axis, label=r"$|\psi(x,y)|^2$")

figure.tight_layout()
plt.show()

## Interpretation

Each product eigenfunction vanishes on all four boundaries. The quantum number $n_x$ controls interior nodal lines parallel to the $y$ axis, while $n_y$ controls interior nodal lines parallel to the $x$ axis. Exchanging $(n_x,n_y)$ rotates the probability-density pattern in the square box and leaves its energy unchanged.

## Exercises

1. Plot $(1,3)$ and $(3,1)$ and count interior nodal lines.
2. Repeat on a rectangular box and retain physical coordinate labels.
3. Compare wavefunction sign structure with probability-density structure.
4. Increase grid density and measure the quadrature normalization residual.
